In [ ]:
import os
import pandas as pd
import glob
import numpy as np

def evaluate_topk_precision(embedding, k, tsv_dir='../results/ppi_results/predicted_probs', interaction_path='../data/ppi/Intra2_interaction.tsv'):
    interaction_df = pd.read_csv(interaction_path, sep='\t', header=None, names=['protein_a', 'protein_b', 'label'])
    labels = interaction_df['label'].values

    # Find all matching prediction files
    pattern = os.path.join(tsv_dir, f"ppi_{embedding}_*.tsv")
    matching_files = sorted(glob.glob(pattern))

    if len(matching_files) == 0:
        raise ValueError(f"No prediction files found for embedding '{embedding}'")

    precisions = []
    false_positives = []

    for file in matching_files:
        with open(file) as f:
            probs = [float(line.strip()) for line in f]
        
        # Pair predicted probs with labels
        paired = list(zip(probs, labels))
        paired.sort(reverse=True, key=lambda x: x[0])  # sort by predicted prob high to low since we want the top-k highest prob

        topk = paired[:k]
        true_positives = sum(1 for _, label in topk if label == 1)
        fp = k - true_positives
        precision = true_positives / k

        precisions.append(precision)
        false_positives.append(fp)

    # Compute mean and std
    avg_precision = np.mean(precisions)
    std_precision = np.std(precisions, ddof=1)
    avg_fp = np.mean(false_positives)
    std_fp = np.std(false_positives, ddof=1)

    return {
        'embedding': embedding,
        'k': k,
        'precision_mean': avg_precision,
        'precision_std': std_precision,
        'false_positives_mean': avg_fp,
        'false_positives_std': std_fp
    }

In [10]:
for embedding in ['1', 'DEF']:
    for k in [10, 100, 1000]:
        print(evaluate_topk_precision(embedding, k))



{'embedding': '1', 'k': 10, 'precision_mean': 1.0, 'precision_std': 0.0, 'false_positives_mean': 0.0, 'false_positives_std': 0.0}
{'embedding': '1', 'k': 100, 'precision_mean': 0.968, 'precision_std': 0.016431676725154998, 'false_positives_mean': 3.2, 'false_positives_std': 1.6431676725154984}
{'embedding': '1', 'k': 1000, 'precision_mean': 0.909, 'precision_std': 0.00244948974278318, 'false_positives_mean': 91.0, 'false_positives_std': 2.449489742783178}
{'embedding': 'DEF', 'k': 10, 'precision_mean': 1.0, 'precision_std': 0.0, 'false_positives_mean': 0.0, 'false_positives_std': 0.0}
{'embedding': 'DEF', 'k': 100, 'precision_mean': 0.994, 'precision_std': 0.005477225575051666, 'false_positives_mean': 0.6, 'false_positives_std': 0.5477225575051662}
{'embedding': 'DEF', 'k': 1000, 'precision_mean': 0.9602, 'precision_std': 0.0013038404810405309, 'false_positives_mean': 39.8, 'false_positives_std': 1.3038404810405297}
